## MAKING AND WRITING MEMORY

In [1]:
# !pip install ollama numpy   # <- uncomment and run once if needed
import ollama, json
import numpy as np
from datetime import datetime, timezone

CHAT_MODEL  = "gemma4:31b-cloud"        
EMBED_MODEL = "embeddinggemma:latest"

In [2]:
def embed(text):
    """Turn text into a meaning-vector using nomic-embed-text."""
    return np.array(ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"], dtype=float)

print("helpers ready: chat(), embed()")

helpers ready: chat(), embed()


### CREATING AN CONVERSATION


In [3]:
import ollama

messages = []   # universal conversation store

def chat_session():
    print("(type 'bye' to end)\n")

    while True:
        user_text = input("You: ")

        if user_text.lower() == "bye":
            break

        # Add user message to conversation history
        messages.append({
            "role": "user",
            "content": user_text
        })

        # Directly call Ollama
        response = ollama.chat(
            model=CHAT_MODEL,
            messages=messages
        )
        # Extract assistant's response
        reply = response["message"]["content"]

        # Add assistant response to conversation history
        messages.append({
            "role": "assistant",
            "content": reply
        })

        print("Assistant:", reply, "\n")


print("chat_session() ready -> type 'bye' to stop")
chat_session()

chat_session() ready -> type 'bye' to stop
(type 'bye' to end)



You:  Hi, I'm Ajay, a data science instructor based in Delhi.


Assistant: Hello Ajay! It's a pleasure to meet you. 

Delhi is such a vibrant hub for technology and education. As a data science instructor, you're right at the center of one of the most exciting fields today.

Whether you're looking to brainstorm new curriculum ideas, need help explaining a complex machine learning concept to your students, or want to stay updated on the latest AI trends, I'm here to help.

Are you focusing on any specific area right now—like Generative AI, Predictive Analytics, or perhaps helping students with their capstone projects? 



You:  Delhi is capital of which country.


Assistant: Delhi is the capital of **India**. 



You:  I'm allergic to peanuts, so keep that in mind for any food suggestions.


... (12 outputs left out)

Assistant: Since you're a data science instructor, you likely encounter this question from students often. While both are powerhouse languages for data science, the fundamental difference lies in their **origin** and **purpose**.

Here is the breakdown:

### 1. Core Purpose
*   **Python:** A **general-purpose** programming language. It was designed to be readable and versatile. You can use it for web development (Django/Flask), automation, software engineering, and AI.
*   **R:** A **statistical** programming language. It was created by statisticians, for statisticians. Its primary purpose is data analysis, complex statistical modeling, and high-quality data visualization.

### 2. Learning Curve and Syntax
*   **Python:** Known for its "clean" and intuitive syntax. It feels closer to the English language, which makes it easier for beginners to pick up.
*   **R:** Can have a steeper learning curve for those without a programming background, as its syntax is specialized for data manipula

You:  exit


Assistant: Goodbye, Ajay! It was great chatting with you. I've saved your preferences—Python over R, peanut allergy, and a preference for step-by-step code explanations.

Good luck with your Context Engineering course for CampusX! Feel free to reach out whenever you're ready to dive back in. Have a great day in Delhi! 



You:  bye


In [4]:
demo_inputs = [
    "Hi, I'm Ajay, a data science instructor based in Delhi.",
    "Delhi is capital of which country.",
    "I'm allergic to peanuts, so keep that in mind for any food suggestions.",
    "Do peanuts contains protein.",
    "Whenever you show me code, explain it step by step — I hate big code dumps.",
    "Last week I started building a Context Engineering course for CampusX.",
    "By the way, what's the capital of France?",
    "I mostly work in Python and prefer it over R.",
    "What is difference between python and R?"
]

In [5]:
messages

[{'role': 'user',
  'content': "Hi, I'm Ajay, a data science instructor based in Delhi."},
 {'role': 'assistant',
  'content': "Hello Ajay! It's a pleasure to meet you. \n\nDelhi is such a vibrant hub for technology and education. As a data science instructor, you're right at the center of one of the most exciting fields today.\n\nWhether you're looking to brainstorm new curriculum ideas, need help explaining a complex machine learning concept to your students, or want to stay updated on the latest AI trends, I'm here to help.\n\nAre you focusing on any specific area right now—like Generative AI, Predictive Analytics, or perhaps helping students with their capstone projects?"},
 {'role': 'user', 'content': 'Delhi is capital of which country.'},
 {'role': 'assistant', 'content': 'Delhi is the capital of **India**.'},
 {'role': 'user',
  'content': "I'm allergic to peanuts, so keep that in mind for any food suggestions."},
 {'role': 'assistant',
  'content': "Noted, Ajay! I have added th

## CREATE STORAGE

In [6]:
class SemanticMemory:
    """Stable FACTS about the user (who they are / what's true).
    Built as a dict. Written by upsert (overwrite by label)."""
    def __init__(self):
        self._facts = {}                 # empty dict is created when the object is made

    def upsert(self, key, value):        # upsert = UPDATE if key exists, INSERT if new
        self._facts[key] = value

    def get(self, key):                  # read ONE fact by its label (None if missing)
        return self._facts.get(key)

    def delete(self, key):               # remove a fact (needed when a fact stops being true)
        self._facts.pop(key, None)

    def get_all(self):
        return dict(self._facts)         # return a COPY of the whole box (no search)


class EpisodicMemory:
    """EVENTS that happened (usually time-stamped).
    Built as a list; each item stores an embedding so it's searchable later."""
    def __init__(self):
        self._events = []                # empty list is created when the object is made

    def add(self, text):
        self._events.append({            # APPEND -- old events are never rewritten
            "text": text,
            "time": datetime.now(timezone.utc).isoformat(),
            "vector": embed(text),  # enbedding
        })

    def get_all(self):                   # matching interface: hand back all events
        return list(self._events)        # return a COPY of the pile


class ProceduralMemory:
    """Learned RULES about how to help this user (how to behave).
    Its OWN store. Behaves like SemanticMemory (small, stable, always-on),
    but kept separate so the three types stay clearly distinct."""
    def __init__(self):
        self._rules = {}

    def upsert(self, key, value):        # same overwrite behaviour as facts
        self._rules[key] = value

    def get(self, key):                  # read ONE rule by its label
        return self._rules.get(key)

    def delete(self, key):               # remove a rule (when a preference changes)
        self._rules.pop(key, None)

    def get_all(self):
        return dict(self._rules)


print("Stores ready: SemanticMemory (facts) | EpisodicMemory (events) | ProceduralMemory (rules)")

Stores ready: SemanticMemory (facts) | EpisodicMemory (events) | ProceduralMemory (rules)


## EXTRACTING MEMORIES THAT WILL BE STORED.

In [7]:
# ---------- Step 1: EXTRACTION (the write decision) ----------

# The instruction that teaches the model WHAT to pull out and HOW to classify it.
# This is static -> it goes in the system role. The changing conversation goes in
# the user role (keeps the system prompt a stable, cacheable prefix).
EXTRACTION_PROMPT = """You extract long-term MEMORY from a conversation — durable facts,
events, and preferences that will help in FUTURE, separate conversations with this user.

Save something only if it will still matter days from now. Ignore greetings, small talk,
one-off questions the user just wanted answered, and anything temporary.

Classify each memory as exactly one type:

- "semantic"  = a lasting FACT about the user (who they are / what's true).
                Stays true tomorrow, no date attached.
                e.g. "Works as a data scientist", "Allergic to peanuts", "Prefers Python"

- "episodic"  = a specific EVENT that happened, tied to a moment in time.
                Describes something that occurred, not something that just is.
                e.g. "Fixed a login bug last Tuesday", "Started a new course last week"

- "procedural"= a RULE about HOW to help this user (their preferred way of being helped).
                An instruction to follow, not a fact about them.
                e.g. "Explain code step by step", "Keep answers short"

Write each memory from the user's perspective: turn "I" into "the user", resolve vague
references, and make each item a single self-contained sentence.

Return ONLY a JSON list. Each item:
  "type":    "semantic" | "episodic" | "procedural"
  "key":     a short snake_case label for semantic/procedural; use "" for episodic
  "content": one short sentence describing the memory

Example
Conversation:
User: Hey! I'm a data scientist, and last week I finally shipped my first ML model to production.
Also please explain code to me step by step, not all at once.
Output:
[
  {"type":"semantic","key":"profession","content":"Works as a data scientist"},
  {"type":"episodic","key":"","content":"Shipped their first ML model to production last week"},
  {"type":"procedural","key":"code_style","content":"Explain code step by step, not all at once"}
]

If nothing is worth remembering, return []."""


def _extract_json(text):
    """Models often wrap JSON in ```json fences or add stray words. Clean it up
    so json.loads() can parse it."""
    t = text.strip()
    if t.startswith("```"):                       # strip a ```json ... ``` fence
        t = t.split("```")[1]
        if t.lower().startswith("json"):
            t = t[4:]
        t = t.strip()
    if not t.startswith("["):                      # fallback: grab first [ ... last ]
        i, j = t.find("["), t.rfind("]")
        if i != -1 and j != -1:
            t = t[i:j + 1]
    return t


def extract_memories(conversation, system_prompt=EXTRACTION_PROMPT):
    """Ask the model to read a conversation and return memory candidates.
    Returns (items, tokens):
      items  -> list of {"type","key","content"} dicts
      tokens -> real prompt token count from Ollama (we care about numbers here)."""
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},                        # static instruction
            {"role": "user",   "content": f"Conversation:\n{conversation}\nOutput:"}  # changing data
        ],
        format="json",   # ask Ollama to guarantee valid JSON -> far fewer parse failures
    )

    reply  = response["message"]["content"]        # the model's text answer
    try:
        items = json.loads(_extract_json(reply))
    except Exception:
        print("Could not parse JSON. Raw reply was:\n", reply)
        items = []

    return items


print("extract_memories() ready")

extract_memories() ready


In [8]:
extract_memories(messages)

[{'type': 'semantic',
  'key': 'user_name',
  'content': "The user's name is Ajay."},
 {'type': 'semantic',
  'key': 'profession',
  'content': 'Works as a data science instructor based in Delhi.'},
 {'type': 'semantic',
  'key': 'health_allergy',
  'content': 'The user is allergic to peanuts.'},
 {'type': 'semantic',
  'key': 'language_preference',
  'content': 'The user mostly works in Python and prefers it over R.'},
 {'type': 'procedural',
  'key': 'code_explanation_style',
  'content': 'Explain code step by step and avoid providing large code dumps.'},
 {'type': 'episodic',
  'key': '',
  'content': 'Started building a Context Engineering course for CampusX last week.'}]

In [9]:
candidates= extract_memories(messages)

In [10]:
candidates

[{'type': 'semantic',
  'key': 'user_identity',
  'content': 'The user is Ajay, a data science instructor based in Delhi.'},
 {'type': 'semantic',
  'key': 'health_allergy',
  'content': 'The user is allergic to peanuts.'},
 {'type': 'procedural',
  'key': 'code_explanation_style',
  'content': 'Explain code step by step and avoid large code dumps.'},
 {'type': 'episodic',
  'key': '',
  'content': 'Started building a Context Engineering course for CampusX last week.'},
 {'type': 'semantic',
  'key': 'language_preference',
  'content': 'The user primarily works in Python and prefers it over R.'}]

## STORING INFORMATION IN RESPECTIVE STORAGE

In [11]:
#FUNCTION TO ROUTE THE MESSAGES TO THE RESPECTIVE STORES
def store_memory(item, semantic, episodic, procedural):
    """Route ONE candidate to the right store, in that store's write style."""
    t = item.get("type")
    if t == "semantic":     
        semantic.upsert(item["key"], item["content"])    # fact  -> semantic store [USING UPSERT FUNCTION]
    elif t == "episodic":   
        episodic.add(item["content"])                     # event -> episodic store
    elif t == "procedural": 
        procedural.upsert(item["key"], item["content"])   # rule  -> procedural store
    else:                   
        print("Unknown type, skipped:", item)


In [12]:
semantic   = SemanticMemory()
episodic   = EpisodicMemory()
procedural = ProceduralMemory()


for item in candidates:
     store_memory(item, semantic, episodic, procedural)
    

In [13]:
def show_memories(semantic, episodic, procedural):
    """Print the THREE kinds of memory, each from its OWN store."""
    print("SEMANTIC   (facts):")
    for k, v in semantic.get_all().items():   print(f"   {k:<12} -> {v}")
    print("EPISODIC   (events):")
    for e in episodic._events:                print(f"   {e['time'][:10]}   {e['text']}")
    print("PROCEDURAL (learned rules):")
    for k, v in procedural.get_all().items(): print(f"   {k:<12} -> {v}")

In [14]:
show_memories(semantic, episodic, procedural)

SEMANTIC   (facts):
   user_identity -> The user is Ajay, a data science instructor based in Delhi.
   health_allergy -> The user is allergic to peanuts.
   language_preference -> The user primarily works in Python and prefers it over R.
EPISODIC   (events):
   2026-08-24   Started building a Context Engineering course for CampusX last week.
PROCEDURAL (learned rules):
   code_explanation_style -> Explain code step by step and avoid large code dumps.


## RETRIVAL  AND INJECTION OF MEMORY 

### SEMATIC AND PROCEDURAL MEMORY

In [15]:
SYSTEM_INSTRUCTION = """You are a helpful personal assistant with memory of this user.

You will be given memory about the user in three parts:
- RULES: how this user wants you to behave. Always follow them.
- FACTS: stable truths about the user. Treat them as true and use them when relevant.
- RELEVANT EVENTS: things that happened before, retrieved for this question. Use them
  only if they help answer the current question; ignore them if they don't fit.

Rely on this memory instead of guessing. If the memory doesn't cover something, answer
normally and don't invent details about the user."""

In [16]:
rules = "\n".join(f"- {v}" for v in procedural.get_all().values()) or "- (none)"
facts = "\n".join(f"- {v}" for v in semantic.get_all().values()) or "- (none)"

In [17]:
rules

'- Explain code step by step and avoid large code dumps.'

In [18]:
SYSTEM_INSTRUCTION =  SYSTEM_INSTRUCTION + "\n\nRULES:\n" + rules + "\n\nFACTS:\n" + facts

In [19]:
print(SYSTEM_INSTRUCTION)

You are a helpful personal assistant with memory of this user.

You will be given memory about the user in three parts:
- RULES: how this user wants you to behave. Always follow them.
- FACTS: stable truths about the user. Treat them as true and use them when relevant.
- RELEVANT EVENTS: things that happened before, retrieved for this question. Use them
  only if they help answer the current question; ignore them if they don't fit.

Rely on this memory instead of guessing. If the memory doesn't cover something, answer
normally and don't invent details about the user.

RULES:
- Explain code step by step and avoid large code dumps.

FACTS:
- The user is Ajay, a data science instructor based in Delhi.
- The user is allergic to peanuts.
- The user primarily works in Python and prefers it over R.


### EPISODIC MEMORY

In [20]:
for i in episodic._events:
    print(i["text"])
    print(i["vector"])

Started building a Context Engineering course for CampusX last week.
[-1.50721356e-01 -4.90900762e-02  3.02529093e-02 -8.88963987e-05
  3.39416228e-02  4.86826710e-02 -5.33122122e-02  4.84469086e-02
  9.39063542e-03  2.68872902e-02 -2.66837478e-02 -2.84964070e-02
  3.77796195e-03 -1.64605714e-02  8.82083103e-02 -1.81059316e-02
  1.27709694e-02  2.39222497e-02  1.49406819e-02  2.08950136e-02
 -3.29268305e-03 -1.57445632e-02 -4.87832315e-02  1.81211382e-02
  1.04424898e-02  4.91537265e-02  4.01502103e-02  1.25972815e-02
  3.34316492e-02 -3.89702320e-02  7.00351298e-02 -2.79374607e-02
  5.74691733e-03  7.47735426e-02 -8.72889534e-03  7.62861222e-02
  5.93349896e-03 -1.70409456e-02  6.42778177e-04 -9.60453413e-03
 -5.32952249e-02  2.74645090e-02 -4.17951792e-02 -2.50990652e-02
  3.92784700e-02 -2.11560261e-02  4.30319866e-04 -7.17717782e-02
  2.71744598e-02 -2.18716450e-02  2.19046623e-02  5.30953472e-03
 -6.74492046e-02  4.07091482e-03 -6.79319575e-02  3.42817716e-02
 -7.62733892e-02  5.0

In [100]:
import numpy as np

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))



# IN VIDEO I NAMED THIS FUNCTION AS RETRIVED_DOCUMETS  BY MISTAKE 
def search_events(query, k=2, min_score=0.45):
    # If the caller passes episodic as the second argument, ignore it
    # and use the global episodic memory instead.
    if not isinstance(k, int):
        k = 2

    events = episodic.get_all()
    hits = []

    if events:
        q = embed(query)

        scored = [
            (cosine(q, e["vector"]), e)
            for e in events
        ]

        scored.sort(
            key=lambda pair: pair[0],
            reverse=True
        )

        hits = [
            (score, event)
            for score, event in scored[:k]
            if score >= min_score
        ]

    return hits

In [101]:
search_events("what food should I avoid?")

[]

In [102]:
search_events("What do you know about the context engineering course??")

[(0.6456312160189205,
  {'text': 'Started building a Context Engineering course for CampusX last week.',
   'time': '2026-08-24T12:17:07.380826+00:00',
   'vector': array([-1.50721356e-01, -4.90900762e-02,  3.02529093e-02, -8.88963987e-05,
           3.39416228e-02,  4.86826710e-02, -5.33122122e-02,  4.84469086e-02,
           9.39063542e-03,  2.68872902e-02, -2.66837478e-02, -2.84964070e-02,
           3.77796195e-03, -1.64605714e-02,  8.82083103e-02, -1.81059316e-02,
           1.27709694e-02,  2.39222497e-02,  1.49406819e-02,  2.08950136e-02,
          -3.29268305e-03, -1.57445632e-02, -4.87832315e-02,  1.81211382e-02,
           1.04424898e-02,  4.91537265e-02,  4.01502103e-02,  1.25972815e-02,
           3.34316492e-02, -3.89702320e-02,  7.00351298e-02, -2.79374607e-02,
           5.74691733e-03,  7.47735426e-02, -8.72889534e-03,  7.62861222e-02,
           5.93349896e-03, -1.70409456e-02,  6.42778177e-04, -9.60453413e-03,
          -5.32952249e-02,  2.74645090e-02, -4.17951792e-0

In [103]:
question="What do you know about the context engineering course??"

In [104]:
retrieved= search_events("What do you know about the context engineering course??")[0][1]['text']

In [105]:
retrieved

'Started building a Context Engineering course for CampusX last week.'

In [106]:
user_question = f"<question>{question}</question><retrieved_chunks>{retrieved}</retrieved_chunks>"

In [107]:
user_question

'<question>What do you know about the context engineering course??</question><retrieved_chunks>Started building a Context Engineering course for CampusX last week.</retrieved_chunks>'

In [108]:
messages = [
    {
        "role": "system",
        "content": SYSTEM_INSTRUCTION
    },
    {
        "role": "user",
        "content": user_question
    }
]

response = ollama.chat(
    model=CHAT_MODEL,
    messages=messages
)

In [109]:
print(response["message"]["content"])

Based on the information I have, you started building a Context Engineering course for CampusX last week.


In [110]:
# ALL TOGETHER

def answer(question):
    """Build the memory message, print it, send it to the model, return the reply.
    Uses the global stores (semantic, episodic, procedural) directly."""

    # ---- SYSTEM section: rules + facts (stable memory) ----
    rules = "\n".join(f"- {v}" for v in procedural.get_all().values()) or "- (none)"
    facts = "\n".join(f"- {v}" for v in semantic.get_all().values()) or "- (none)"
    system = {"role": "system", "content":
        SYSTEM_INSTRUCTION + "\n\nRULES:\n" + rules + "\n\nFACTS:\n" + facts}

    # ---- USER section: the question, with events added ONLY if any are relevant ----
    events = search_events(question, episodic)
    if events:                                          # only when there's a real match
        event_text = "\n".join(f"- {e['text']}" for _, e in events)
        user_content = "RELEVANT EVENTS:\n" + event_text + "\n\nQuestion: " + question
    else:                                               # no relevant event -> keep it clean
        user_content = question
    user = {"role": "user", "content": user_content}

    messages = [system, user]

    # ---- call the LLM ----
    reply = ollama.chat(model=CHAT_MODEL, messages=messages)["message"]["content"]
    print("\nASSISTANT:\n" + reply)

In [111]:
answer("What snack can I have?")


ASSISTANT:
Since you have a peanut allergy, you should avoid any snacks containing peanuts or those processed in facilities that handle them. 

Depending on what you're in the mood for, here are a few safe suggestions:

**Savory Options:**
*   **Roasted Chickpeas:** A great high-protein alternative to nuts.
*   **Popcorn:** Classic and easy, just check the seasoning if it's store-bought.
*   **Fresh Fruit & Cheese:** Slices of apple or pear with a piece of cheddar or brie.
*   **Hummus and Veggies:** Carrots, cucumber, and bell peppers with hummus.

**Sweet Options:**
*   **Greek Yogurt:** Topped with honey and fresh berries.
*   **Fresh Fruit:** Mangoes or grapes are always refreshing.
*   **Dark Chocolate:** Just be sure to check the label for "may contain peanuts" warnings.

**A quick tip:** Since you're in Delhi, if you're ordering from a local spot, double-check that they haven't used peanut oil for frying!


In [112]:
answer("What course am I working on?")  


ASSISTANT:
You are currently working on a Context Engineering course for CampusX.


In [113]:
answer("Write me a function to reverse a string.")  


ASSISTANT:
Since you primarily work in Python, the most efficient way to do this is by using **string slicing**. 

Here is the function:

```python
def reverse_string(text):
    return text[::-1]
```

### Step-by-Step Explanation:

1.  **The Slicing Syntax**: In Python, the slice syntax is `[start:stop:step]`.
2.  **Omitting Start and Stop**: By leaving the `start` and `stop` indices empty (the first two colons), Python defaults to the beginning and the end of the entire string.
3.  **The Step Value**: The `step` value is set to `-1`. This tells Python to traverse the string backward, one character at a time.
4.  **The Result**: As a result, the function returns a new string that is the exact reverse of the input.

**Example Usage:**
```python
print(reverse_string("Delhi"))  # Output: ihleD
```


## RUNNING AGENT 

In [114]:
def run_agent():
    """Chat loop with memory. Keeps the conversation history across turns (short-term
    memory) on top of the long-term stores. Uses the global stores directly."""
    history = []
    print("Chat started. Type 'exit' to stop.\n")

    while True:
        question = input("You: ").strip()
        if question.lower() in ("exit", "quit", ""):
            print("Chat ended.")
            break

        # ---- SYSTEM: rules + facts (long-term, stable) ----
        rules = "\n".join(f"- {v}" for v in procedural.get_all().values()) or "- (none)"
        facts = "\n".join(f"- {v}" for v in semantic.get_all().values()) or "- (none)"
        system = {"role": "system", "content":
            SYSTEM_INSTRUCTION + "\n\nRULES:\n" + rules + "\n\nFACTS:\n" + facts}

        # ---- USER: the question, with events added only if relevant ----
        events = search_events(question, episodic)
        if events:
            event_text = "\n".join(f"- {e['text']}" for _, e in events)
            user_content = "RELEVANT EVENTS:\n" + event_text + "\n\nQuestion: " + question
        else:
            user_content = question
        user = {"role": "user", "content": user_content}

        # ---- messages = system + conversation so far + this turn ----
        messages = [system] + history + [user]
        reply = ollama.chat(model=CHAT_MODEL, messages=messages)["message"]["content"]
        print("Assistant:", reply, "\n")

        # ---- store the plain exchange in history (short-term memory) ----
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": reply})

In [115]:
run_agent()

Chat started. Type 'exit' to stop.



You:  Hi there


Assistant: Hello Ajay! How can I help you today? 



You:  What snack should I grab


Assistant: Since you have a peanut allergy, you'll want to be careful with many store-bought snacks. Depending on what you're in the mood for, here are a few safe suggestions:

**Savory:**
*   **Popcorn:** Usually a safe bet, though always check the label if it's a specialty flavor.
*   **Hummus and Carrot Sticks:** A healthy, protein-rich option.
*   **Cheese and Crackers:** A classic choice.

**Sweet:**
*   **Fresh Fruit:** Apples, bananas, or grapes are always safe.
*   **Greek Yogurt:** Great with some honey or fresh berries.
*   **Dark Chocolate:** Just be sure to check for "may contain traces of peanuts" warnings on the packaging.

**Pro Tip:** Since you're in Delhi, if you're ordering from a local spot, just double-check that they don't use peanut oil for frying! 



You:  Write me a function to reverse a string


... (5 outputs left out)

You:  What did I just ask you?


Assistant: You just asked me, "What's the capital of Japan?" 



You:  exit


Chat ended.


In [ ]:
Hi there
What snack should I grab?
Write me a function to reverse a string
What course am I working on?
What's the capital of Japan?
What did I just ask you?